In [13]:
import duckdb

In [14]:
# Initialize database path
db_file = '../olist_medallion_architecture.db'

# Connect to database file
conn = duckdb.connect(db_file, read_only = True)

In [15]:
# Verify views exist in data dictionary
conn.query("""
    SELECT view_name
    FROM duckdb_views() 
    WHERE NOT internal
        AND view_name LIKE 'silver%'
    ORDER BY
        view_name;
""")

view_nameVARCHAR
silver_customers
silver_order_items
silver_orders
silver_payments
silver_products
silver_reviews
silver_sellers


### Testing UDF at view call level

In [9]:
from deep_translator import MyMemoryTranslator
from duckdb.sqltypes import VARCHAR

# User-defined functions (UDF) definitions 
def translate_to_english (text: str) -> str:
    en_text = MyMemoryTranslator(source = 'pt-PT', target = 'en-US').translate(text)
    return en_text

# Register UDF in duckdb
try:
    conn.remove_function('translate_to_english')
except Exception:
    pass

conn.create_function('translate_to_english', translate_to_english, [VARCHAR], VARCHAR)

In [10]:
# Evaluate view records
conn.query("""
    SELECT *
    FROM silver_reviews
    WHERE comment IS NOT NULL
    LIMIT 5;
""")

InvalidInputException: Invalid Input Error: Python exception occurred while executing the UDF: TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

InvalidInputException: Invalid Input Error: Python exception occurred while executing the UDF: TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

In [16]:
conn.close()